# ENAPRES 2025 — Base de hogares y programas sociales


## 1. Instalar bibliotecas (primera ejecución)


In [ ]:
%pip install -q numpy pandas pyreadstat openpyxl

## 2. Importaciones y configuración del procesamiento


In [ ]:
from __future__ import annotations

import json
import re
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Optional

from openpyxl import Workbook
from openpyxl.cell import WriteOnlyCell
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

import numpy as np
import pandas as pd

try:
    import pyreadstat
except ImportError:
    pyreadstat = None


In [ ]:
@dataclass
class Configuracion:
    directorio_entrada: Path = Path('/mnt/data') if Path('/mnt/data').exists() else Path.cwd()
    directorio_salida: Optional[Path] = None

    resfin_permitidos: tuple[int, ...] = (1,)

    solo_urbano: bool = False
    edad_minima_valida: int = 0
    edad_maxima_valida: int = 120
    incluir_modulo_500: bool = False

    excel_filas_maximas_por_hoja: int = 1_048_575


CLAVE_HOGAR = ['ANIO', 'MES', 'CONGLOMERADO', 'VIVIENDA', 'HOGAR']
CLAVE_PERSONA = CLAVE_HOGAR + ['P201']

PROGRAMAS = {
    'qali_warma_desayuno': 'P306_1',
    'qali_warma_almuerzo': 'P306_2',
    'cuna_mas_guarderia': 'P306_3',
    'cuna_mas_acompanamiento': 'P306_4',
    'pension_65': 'P306_6',
    'comedor_popular': 'P306_7',
    'vaso_de_leche': 'P306_8',
    'juntos': 'P306_9',
    'sis': 'P306_10',
    'otro_programa': 'P306_11',
}
NODOS_ORIGINALES = [f'prog_{nombre}' for nombre in PROGRAMAS]
NODOS = [
    'prog_qali_warma',
    'prog_cuna_mas',
    'prog_pension_65',
    'prog_juntos',
    'prog_vaso_de_leche',
    'prog_comedor_popular',
    'prog_sis',
]
ACTIVOS = [f'P189_{i}' for i in range(1, 18)]

COLUMNAS_CAP200 = CLAVE_PERSONA + [
    'RESFIN', 'AREA', 'CCDD', 'NOMBREDD', 'REGIONNATU', 'P202C', 'P203A',
    'P204', 'P205', 'P206', 'P207', 'P208_A', 'FACTOR',
]
COLUMNAS_CAP300 = CLAVE_PERSONA + [
    'RESFIN', 'P300C_A', 'P300C_C', 'P300_1',
    *[f'P301_{i}' for i in range(1, 8)],
    *PROGRAMAS.values(), 'P307', 'P308', 'P312', 'P313',
]
COLUMNAS_CAP100E = CLAVE_HOGAR + [
    'RESFIN', 'AREA', 'P101', 'P102', 'P103', 'P104', 'P105', 'P106',
    'P113', 'P115A', 'P135', 'P136', 'P138', 'P170', 'P179',
    *ACTIVOS, 'P189_18', 'ESTRATO', 'FACTOR',
]
COLUMNAS_CAP500 = CLAVE_PERSONA + ['RESFIN', 'P500_A', 'P501', 'P502']

ESQUEMA_LECTURA = {
    '200': ('200', COLUMNAS_CAP200),
    '300C': ('300', COLUMNAS_CAP300),
    '100E': ('100E', COLUMNAS_CAP100E),
    '500': ('500', COLUMNAS_CAP500),
}

COLUMNAS_FINALES = ['ANIO', 'MES', 'CONGLOMERADO', 'VIVIENDA', 'HOGAR', 'factor_hogar', 'prog_qali_warma', 'prog_cuna_mas', 'prog_pension_65', 'prog_juntos', 'prog_vaso_de_leche', 'prog_comedor_popular', 'prog_sis', 'edad_jefe', 'sexo_jefe', 'tamano_hogar', 'educacion_jefe_4niveles', 'lengua_materna_jefe', 'jefe_pertenece_pueblo_indigena', 'area', 'codigo_departamento', 'region_natural', 'agua_red_publica', 'saneamiento_red_publica', 'electricidad_red_publica', 'indice_activos_pca', 'n_ninos_0_11', 'n_adolescentes_12_17', 'n_adultos_65mas', 'n_mujeres', 'hogar_multigeneracional', 'hogar_con_discapacidad', 'n_integrantes_con_discapacidad', 'internet_en_hogar', 'hogar_con_dispositivo_digital']
assert len(COLUMNAS_FINALES) == 35 and len(set(COLUMNAS_FINALES)) == 35


## 3. Lectura de archivos `.sav`, claves y filtros de calidad


In [ ]:
def numerica(serie: pd.Series) -> pd.Series:
    return pd.to_numeric(serie, errors='coerce')


def yn(serie: pd.Series) -> pd.Series:
    s = numerica(serie)
    return pd.Series(np.select([s.eq(1), s.eq(2)], [1.0, 0.0], default=np.nan),
                     index=serie.index, dtype='float64')


def seleccion(serie: pd.Series) -> pd.Series:
    s = numerica(serie)
    return pd.Series(np.select([s.eq(1), s.eq(0)], [1.0, 0.0], default=np.nan),
                     index=serie.index, dtype='float64')


def clave_como_texto(serie: pd.Series) -> pd.Series:
    r = serie.astype('string').str.strip().str.replace(r'\.0+$', '', regex=True)
    r = r.replace({'': pd.NA, 'nan': pd.NA, 'NaN': pd.NA, 'None': pd.NA, '<NA>': pd.NA})
    return r


def estandarizar_claves(datos: pd.DataFrame, columnas: list[str], nombre: str) -> pd.DataFrame:
    faltantes = [c for c in columnas if c not in datos.columns]
    if faltantes:
        raise ValueError(f'{nombre}: faltan identificadores {faltantes}')
    datos = datos.copy()
    for c in columnas:
        datos[c] = clave_como_texto(datos[c])
    n_nulas = int(datos[columnas].isna().any(axis=1).sum())
    if n_nulas:
        raise ValueError(f'{nombre}: {n_nulas} filas con llaves incompletas. No se unen.')
    return datos


def verificar_unicidad(datos: pd.DataFrame, llave: list[str], nombre: str) -> None:
    dup = datos.duplicated(llave, keep=False)
    if dup.any():
        ejemplos = datos.loc[dup, llave].head(5).to_dict('records')
        raise ValueError(f'{nombre}: llave NO unica en {int(dup.sum())} filas. Ejemplos: {ejemplos}')


def descubrir_archivos(entrada: Path, tag: str) -> list[Path]:
    expresion = re.compile(rf'^CAP_{re.escape(tag)}(?:_|$)', flags=re.IGNORECASE)
    return sorted((f for f in entrada.rglob('*') if f.is_file() and
                   f.suffix.lower() == '.sav' and expresion.search(f.stem)),
                  key=lambda p: str(p).lower())


def cargar_capitulo(config: Configuracion, nombre: str, auditoria: dict) -> pd.DataFrame:
    if pyreadstat is None:
        raise ImportError('No se instalo pyreadstat. En Jupyter ejecute: %pip install pyreadstat pyarrow')
    tag, seleccionadas = ESQUEMA_LECTURA[nombre]
    archivos = descubrir_archivos(Path(config.directorio_entrada), tag)
    if not archivos:
        raise FileNotFoundError(f'No se encontro CAP_{tag}_*.sav en {config.directorio_entrada}')
    piezas = []
    for ruta in archivos:
        tabla, metadata = pyreadstat.read_sav(
            str(ruta), usecols=seleccionadas, apply_value_formats=False,
            formats_as_category=False, user_missing=False,
        )
        tabla.columns = [str(c).strip().upper() for c in tabla.columns]
        faltantes = set(seleccionadas) - set(tabla.columns)
        if faltantes:
            raise KeyError(f'{ruta.name}: faltan variables exigidas: {sorted(faltantes)}')
        inicial = len(tabla)
        tabla = tabla.loc[numerica(tabla['RESFIN']).isin(config.resfin_permitidos)].copy()
        auditoria['lectura'].append({
            'modulo': nombre, 'archivo': ruta.name, 'filas_totales': int(inicial),
            'filas_post_resfin': int(len(tabla)),
            'resfin_permitidos': list(config.resfin_permitidos),
        })
        piezas.append(tabla)
    unido = pd.concat(piezas, ignore_index=True)
    llave = CLAVE_HOGAR if nombre == '100E' else CLAVE_PERSONA
    unido = estandarizar_claves(unido, llave, f'CAP{nombre}')
    return unido


def incorporar_persona_objetivo(datos: pd.DataFrame, campo: str, nombre: str, auditoria: dict) -> pd.DataFrame:
    out = datos.copy()
    objetivo = numerica(out[campo])
    objetivo_valido = objetivo.ge(1) & objetivo.le(80)
    objetivo_txt = clave_como_texto(out[campo])
    diferencias = objetivo_valido & objetivo_txt.ne(out['P201']).fillna(False)
    auditoria['personas_objetivo'][nombre] = {
        'diferencias_P201_vs_objetivo': int(diferencias.sum()),
        'objetivo_invalido_o_ausente': int((~objetivo_valido).sum()),
    }
    out['P201'] = out['P201'].where(~objetivo_valido, objetivo_txt)
    return out


def _uno_si_algun_positivo_otro_cero(serie: pd.Series) -> float:
    s = pd.to_numeric(serie, errors='coerce')
    if s.eq(1).any():
        return 1.0
    if s.notna().all() and len(s) and s.eq(0).all():
        return 0.0
    return np.nan


## 4. Residentes habituales, indicadores demográficos y composición familiar


In [ ]:
def construir_padron(cap200: pd.DataFrame, config: Configuracion, auditoria: dict):
    d = cap200.copy()
    for c in ['P204', 'P205', 'P206', 'P207', 'P208_A', 'P202C', 'P203A',
              'AREA', 'REGIONNATU', 'FACTOR']:
        d[c] = numerica(d[c])
    residente = ((d['P204'] == 1) & (d['P205'] == 2)) | ((d['P204'] == 2) & (d['P206'] == 1))
    edad_valida = d['P208_A'].between(config.edad_minima_valida, config.edad_maxima_valida)
    conservar = residente & edad_valida
    if config.solo_urbano:
        conservar &= d['AREA'].eq(1)
    auditoria['filtros']['cap200'] = {
        'personas_antes': int(len(d)), 'residentes_habituales': int(residente.sum()),
        'residentes_edad_valida': int((residente & edad_valida).sum()),
        'personas_finales': int(conservar.sum()),
        'no_urban_only': not config.solo_urbano,
    }
    d = d.loc[conservar].copy()
    verificar_unicidad(d, CLAVE_PERSONA, 'CAP200 residentes')
    if d.empty:
        raise ValueError('CAP200 no tiene residentes despues del filtro')

    edad = d['P208_A']
    sexo = d['P207']
    d['_ninos_0_11'] = edad.between(0, 11).astype('int8')
    d['_adolescentes_12_17'] = edad.between(12, 17).astype('int8')
    d['_menores_18'] = edad.lt(18).astype('int8')
    d['_menores_5'] = edad.between(0, 4).astype('int8')
    d['_menores_3'] = edad.between(0, 2).astype('int8')
    d['_adultos_18_64'] = edad.between(18, 64).astype('int8')
    d['_adultos_65mas'] = edad.ge(65).astype('int8')
    d['_mujeres'] = sexo.eq(2).astype('int8')
    d['_sexo_sin_dato'] = (~sexo.isin([1, 2])).astype('int8')
    d['_nucleos'] = d['P203A'].where(d['P203A'].gt(0))

    g = d.groupby(CLAVE_HOGAR, dropna=False, sort=False)
    hh = g.agg(
        tamano_hogar=('P201', 'size'),
        edad_promedio=('P208_A', 'mean'),
        n_ninos_0_11=('_ninos_0_11', 'sum'),
        n_adolescentes_12_17=('_adolescentes_12_17', 'sum'),
        n_menores_18=('_menores_18', 'sum'),
        n_menores_5=('_menores_5', 'sum'),
        n_menores_3=('_menores_3', 'sum'),
        n_adultos_18_64=('_adultos_18_64', 'sum'),
        n_adultos_65mas=('_adultos_65mas', 'sum'),
        n_mujeres=('_mujeres', 'sum'),
        n_sexo_sin_dato=('_sexo_sin_dato', 'sum'),
        n_nucleos_familiares=('_nucleos', 'nunique'),
        area=('AREA', 'first'),
        codigo_departamento=('CCDD', 'first'),
        departamento=('NOMBREDD', 'first'),
        region_natural=('REGIONNATU', 'first'),
        factor_cap200=('FACTOR', 'first'),
    ).reset_index()
    hh['prop_mujeres'] = hh['n_mujeres'] / hh['tamano_hogar']
    hh.loc[hh['n_sexo_sin_dato'] > 0, 'prop_mujeres'] = np.nan
    hh['hogar_con_ninos'] = (hh['n_ninos_0_11'] > 0).astype('int8')
    hh['hogar_con_adolescentes'] = (hh['n_adolescentes_12_17'] > 0).astype('int8')
    hh['hogar_con_adultos_mayores'] = (hh['n_adultos_65mas'] > 0).astype('int8')
    hh['hogar_intergeneracional'] = ((hh['n_menores_18'] > 0) &
                                     (hh['n_adultos_65mas'] > 0)).astype('int8')
    hh['hogar_multigeneracional'] = ((hh['n_menores_18'] > 0) &
                                     (hh['n_adultos_18_64'] > 0) &
                                     (hh['n_adultos_65mas'] > 0)).astype('int8')
    hh['region_natural'] = pd.to_numeric(hh['region_natural'], errors='coerce')
    hh['area'] = pd.to_numeric(hh['area'], errors='coerce')
    hh['factor_cap200'] = pd.to_numeric(hh['factor_cap200'], errors='coerce')

    jefes = d.loc[d['P202C'].eq(1), CLAVE_PERSONA + ['P207', 'P208_A']].copy()
    verificar_unicidad(jefes, CLAVE_HOGAR, 'Jefe/jefa del hogar')
    jefes = jefes.rename(columns={'P207': 'sexo_jefe', 'P208_A': 'edad_jefe',
                                  'P201': 'orden_jefe'})
    hh = hh.merge(jefes, how='left', on=CLAVE_HOGAR, validate='one_to_one', sort=False)
    auditoria['filtros']['hogares_sin_jefe_identificado'] = int(hh['orden_jefe'].isna().sum())
    return d, hh


## 5. Programas sociales por hogar y universo de edad


In [ ]:
def agregar_cap300_a_hogares(personas: pd.DataFrame, cap300: pd.DataFrame,
                            hh: pd.DataFrame, config: Configuracion, auditoria: dict):
    d = incorporar_persona_objetivo(cap300, 'P300C_A', 'CAP300C', auditoria)
    verificar_unicidad(d, CLAVE_PERSONA, 'CAP300C')
    solo = CLAVE_PERSONA + ['P300_1', *[f'P301_{i}' for i in range(1, 8)],
                            *PROGRAMAS.values(), 'P307', 'P308', 'P312', 'P313']
    d = d[solo].copy()
    a = personas.merge(d, how='left', on=CLAVE_PERSONA, validate='one_to_one',
                       indicator='_origen_cap300', sort=False)
    a['_registro_cap300'] = a['_origen_cap300'].eq('both').astype('int8')
    a['_discapacidad'] = pd.DataFrame({
        k: yn(a[f'P301_{k}']) for k in range(1, 8)
    }, index=a.index).apply(_uno_si_algun_positivo_otro_cero, axis=1)
    a['_pertenencia_indigena'] = yn(a['P313'])
    a['_no_tiene_dni'] = yn(a['P300_1']).map({0.0: 1.0, 1.0: 0.0})

    grp = a.groupby(CLAVE_HOGAR, dropna=False, sort=False)
    c = grp.agg(
        n_registros_cap300=('_registro_cap300', 'sum'),
        n_personas_discapacidad_observada=('_discapacidad', 'count'),
        n_personas_discapacidad_confirmada=('_discapacidad', lambda x: int(x.eq(1).sum())),
        n_integrantes_indigenas_observados=('_pertenencia_indigena', 'count'),
        n_integrantes_indigenas_confirmados=('_pertenencia_indigena', lambda x: int(x.eq(1).sum())),
        n_integrantes_sin_dni_confirmados=('_no_tiene_dni', lambda x: int(x.eq(1).sum())),
    ).reset_index()
    c = c.merge(hh[CLAVE_HOGAR + ['tamano_hogar']], on=CLAVE_HOGAR,
                how='left', validate='one_to_one')
    c['cobertura_cap300_completa'] = (
        (c['n_registros_cap300'] == c['tamano_hogar'])
    ).astype('int8')

    UNIVERSO_EDAD_PROGRAMAS = {
        'qali_warma_desayuno': (0, 17),
        'qali_warma_almuerzo': (0, 17),
        'cuna_mas_guarderia': (0, 17),
        'cuna_mas_acompanamiento': (0, 17),
        'pension_65': (65, config.edad_maxima_valida),
    }

    def booleano_presencia(nombre: str, campo: str) -> None:
        raw = numerica(a[campo])
        edad = numerica(a['P208_A'])
        intervalo = UNIVERSO_EDAD_PROGRAMAS.get(nombre)
        elegible = (edad.between(*intervalo) if intervalo is not None
                    else pd.Series(True, index=a.index))

        a[f'_si_{nombre}'] = raw.eq(1).astype('int8')
        a[f'_eleg_{nombre}'] = elegible.astype('int8')
        a[f'_contestado_{nombre}'] = (elegible & raw.isin([1, 2])).astype('int8')
        resumen = grp[[f'_si_{nombre}', f'_eleg_{nombre}',
                       f'_contestado_{nombre}']].sum().reset_index()

        n_si = resumen[f'_si_{nombre}']
        n_elegibles = resumen[f'_eleg_{nombre}']
        n_respondieron = resumen[f'_contestado_{nombre}']
        completo = n_elegibles.eq(n_respondieron)

        presencia = np.select([n_si.gt(0), completo],
                              [1.0, 0.0], default=np.nan)
        c[f'prog_{nombre}'] = presencia.astype('float64')
        c[f'n_beneficiarios_{nombre}'] = n_si.astype('float64').to_numpy()
        auditoria.setdefault('programas', {})[nombre] = {
            'hogares_si': int(np.count_nonzero(presencia == 1.0)),
            'hogares_no': int(np.count_nonzero(presencia == 0.0)),
            'hogares_indeterminados': int(np.isnan(presencia).sum()),
            'hogares_sin_personas_en_universo': int(n_elegibles.eq(0).sum()),
            'personas_si': int(n_si.sum()),
        }

    for nombre, campo in PROGRAMAS.items():
        booleano_presencia(nombre, campo)

    print('Resumen verificado de la agregacion de programas por hogar:')
    for nombre, s in auditoria['programas'].items():
        print(f"  {nombre:29} si={s['hogares_si']:>6,} "
              f"no={s['hogares_no']:>6,} "
              f"indeterminados={s['hogares_indeterminados']:>6,}")

    def unir_modalidades(primera: str, segunda: str) -> np.ndarray:
        a_ = c[primera]
        b_ = c[segunda]
        return np.select(
            [a_.eq(1) | b_.eq(1), a_.eq(0) & b_.eq(0)],
            [1.0, 0.0], default=np.nan
        )

    c['prog_qali_warma'] = unir_modalidades(
        'prog_qali_warma_desayuno', 'prog_qali_warma_almuerzo'
    )
    c['prog_cuna_mas'] = unir_modalidades(
        'prog_cuna_mas_guarderia', 'prog_cuna_mas_acompanamiento'
    )
    c['n_programas_hogar'] = c[NODOS].sum(axis=1, min_count=len(NODOS))
    c['hogar_con_discapacidad'] = grp['_discapacidad'].agg(
        _uno_si_algun_positivo_otro_cero
    ).to_numpy(dtype='float64')
    c['n_integrantes_con_discapacidad'] = c['n_personas_discapacidad_confirmada'].where(
        c['n_personas_discapacidad_observada'].eq(c['tamano_hogar']), np.nan
    )
    c['hogar_con_integrante_indigena'] = grp['_pertenencia_indigena'].agg(
        _uno_si_algun_positivo_otro_cero
    ).to_numpy(dtype='float64')
    c['hogar_con_integrante_sin_dni'] = grp['_no_tiene_dni'].agg(
        _uno_si_algun_positivo_otro_cero
    ).to_numpy(dtype='float64')

    j = a.loc[a['P202C'].eq(1), CLAVE_HOGAR + ['P307', 'P308', 'P312', 'P313']].copy()
    verificar_unicidad(j, CLAVE_HOGAR, 'Jefe CAP300C')
    j = j.rename(columns={
        'P307': 'lengua_materna_jefe',
        'P308': 'nivel_educativo_jefe',
        'P312': 'autoidentificacion_etnica_jefe',
        'P313': 'pertenencia_indigena_jefe_raw',
    })
    c = c.merge(j, on=CLAVE_HOGAR, how='left', validate='one_to_one')
    for k in ['lengua_materna_jefe', 'nivel_educativo_jefe',
              'autoidentificacion_etnica_jefe']:
        c[k] = numerica(c[k]).where(numerica(c[k]).gt(0))
    lengua = c['lengua_materna_jefe']
    c['jefe_lengua_originaria'] = np.where(lengua.between(1, 9), 1.0,
                                            np.where(lengua.between(10, 13), 0.0, np.nan))
    edu = c['nivel_educativo_jefe']
    mapa_edu = {1: 1, 2: 1, 3: 1, 4: 1, 7: 1, 5: 2, 6: 2,
                8: 3, 10: 3, 9: 4, 11: 4, 12: 4}
    c['educacion_jefe_4niveles'] = edu.map(mapa_edu).astype('float64')
    c['jefe_pertenece_pueblo_indigena'] = yn(c['pertenencia_indigena_jefe_raw'])
    c['jefe_autoidentificacion_indigena'] = np.where(
        c['autoidentificacion_etnica_jefe'].between(1, 4), 1.0,
        np.where(c['autoidentificacion_etnica_jefe'].between(5, 8), 0.0, np.nan)
    )
    auditoria['uniones']['CAP300C'] = {
        'registros_cap200_residentes': int(len(personas)),
        'registros_cap300_efectivamente_enlazados': int(a['_registro_cap300'].sum()),
        'hogares_cobertura_completa': int(c['cobertura_cap300_completa'].sum()),
        'hogares_cobertura_incompleta': int((c['cobertura_cap300_completa'] == 0).sum()),
    }
    return c.drop(columns=['tamano_hogar']), a


## 6. Vivienda, servicios básicos, bienes, conectividad e índice de activos PCA


In [ ]:
def construir_hogares_cap100e(cap100e: pd.DataFrame, auditoria: dict):
    df = cap100e.copy()
    verificar_unicidad(df, CLAVE_HOGAR, 'CAP100E')
    salida = df[CLAVE_HOGAR].copy()
    p105 = numerica(df['P105'])
    p106 = numerica(df['P106'])
    salida['habitaciones_vivienda'] = p105.where(p105.gt(0))
    salida['tenencia_vivienda'] = p106.where(p106.between(1, 8))
    salida['vivienda_propia'] = np.where(p106.isin([2, 3, 4]), 1.0,
                                       np.where(p106.isin([1, 5, 6, 7, 8]), 0.0, np.nan))
    p101 = numerica(df['P101'])
    p102 = numerica(df['P102'])
    p103 = numerica(df['P103'])
    p104 = numerica(df['P104'])
    salida['vivienda_material_precario_aprox'] = np.where(
        p101.isin([5, 6, 7]) | p102.isin([8, 9]) |
        p103.eq(6) | p104.isin([6, 7, 8]), 1.0,
        np.where(p101.notna() & p102.notna() & p103.notna() & p104.notna(), 0.0, np.nan)
    )
    salida['electricidad_red_publica'] = yn(df['P113'])
    salida['electricidad_diaria'] = yn(df['P115A'])
    fuente_agua = numerica(df['P135'])
    salida['agua_red_publica'] = np.where(fuente_agua.isin([1, 2, 3, 9]), 1.0,
                                          np.where(fuente_agua.between(1, 9), 0.0, np.nan))
    salida['agua_reportada_potable'] = yn(df['P136'])
    p138 = numerica(df['P138'])
    salida['agua_todos_los_dias'] = np.where(p138.eq(1), 1.0,
                                            np.where(p138.isin([2, 3]), 0.0, np.nan))
    saneamiento = numerica(df['P170'])
    salida['saneamiento_red_publica'] = np.where(saneamiento.isin([1, 2, 9]), 1.0,
                                               np.where(saneamiento.between(1, 10), 0.0, np.nan))
    salida['recoleccion_municipal_basura'] = yn(df['P179'])
    salida['n_servicios_basicos'] = salida[
        ['agua_red_publica', 'saneamiento_red_publica', 'electricidad_red_publica']
    ].sum(axis=1, min_count=3)
    salida['tres_servicios_basicos'] = np.where(
        salida['n_servicios_basicos'].notna(),
        (salida['n_servicios_basicos'] == 3).astype('float64'), np.nan
    )
    salida['area_cap100e'] = numerica(df['AREA'])
    salida['estrato_geografico_reportado'] = numerica(df['ESTRATO'])
    salida['factor_hogar'] = numerica(df['FACTOR']).where(numerica(df['FACTOR']).gt(0))

    for codigo in ACTIVOS:
        salida['activo_' + codigo.lower()] = seleccion(df[codigo])
    columnas_activos = ['activo_' + codigo.lower() for codigo in ACTIVOS]
    salida['n_bienes'] = salida[columnas_activos].sum(axis=1, min_count=len(ACTIVOS))
    sin_bienes_reportado = seleccion(df['P189_18']).eq(1)
    auditoria['activos'] = {
        'filas_con_nada_seleccionado_y_al_menos_un_activo': int(
            (sin_bienes_reportado & salida['n_bienes'].gt(0)).sum()
        ),
        'activos_para_pca': columnas_activos,
    }
    salida['internet_en_hogar'] = salida['activo_p189_14']
    salida['hogar_con_computadora'] = salida['activo_p189_9']
    salida['hogar_con_celular_internet'] = salida['activo_p189_11']
    salida['hogar_con_celular'] = pd.concat([
        salida['activo_p189_10'], salida['activo_p189_11']
    ], axis=1).max(axis=1, skipna=False)
    salida['hogar_con_dispositivo_digital'] = pd.concat([
        salida['activo_p189_9'], salida['activo_p189_11']
    ], axis=1).max(axis=1, skipna=False)

    salida['indice_activos_pca'], pca_detalles = pca_ponderado(
        salida[columnas_activos], salida['factor_hogar']
    )
    auditoria['pca'] = pca_detalles
    salida['quintil_activos_ponderado'] = quintiles_ponderados(
        salida['indice_activos_pca'], salida['factor_hogar']
    )
    return salida


def pca_ponderado(X: pd.DataFrame, pesos: pd.Series):
    datos = X.apply(numerica).astype(float)
    w = numerica(pesos).astype(float)
    valido = datos.notna().all(axis=1) & w.gt(0) & np.isfinite(w)
    if valido.sum() < 3:
        raise ValueError('PCA: menos de tres hogares con 17 bienes y peso positivo.')
    xs = datos.loc[valido].copy()
    ws = w.loc[valido].to_numpy(dtype='float64')
    vars_usables, medias, desviaciones = [], {}, {}
    for var in xs.columns:
        v = xs[var].to_numpy(dtype='float64')
        media = float(np.average(v, weights=ws))
        varianza = float(np.average((v - media) ** 2, weights=ws))
        if varianza > 0 and np.isfinite(varianza):
            vars_usables.append(var)
            medias[var] = media
            desviaciones[var] = float(np.sqrt(varianza))
    if len(vars_usables) < 2:
        raise ValueError('PCA: menos de dos bienes con varianza ponderada positiva.')
    z = np.column_stack([
        (xs[v].to_numpy(dtype=float) - medias[v]) / desviaciones[v]
        for v in vars_usables
    ])
    cov = (z * ws[:, None]).T @ z / ws.sum()
    autovalores, autovectores = np.linalg.eigh(cov)
    orden = np.argsort(autovalores)[::-1]
    autovalores = autovalores[orden]
    cargas = autovectores[:, orden[0]]
    puntuaciones = z @ cargas
    totales = xs[vars_usables].sum(axis=1).to_numpy(dtype=float)
    x_c = puntuaciones - np.average(puntuaciones, weights=ws)
    y_c = totales - np.average(totales, weights=ws)
    producto = np.average(x_c * y_c, weights=ws)
    if producto < 0:
        puntuaciones *= -1
        cargas *= -1
    media_p = float(np.average(puntuaciones, weights=ws))
    dt_p = float(np.sqrt(np.average((puntuaciones - media_p) ** 2, weights=ws)))
    if dt_p <= 0 or not np.isfinite(dt_p):
        raise ValueError('PCA: la puntuacion no tiene dispersion valida.')
    puntuaciones = (puntuaciones - media_p) / dt_p
    scores = pd.Series(np.nan, index=datos.index, dtype='float64')
    scores.loc[valido] = puntuaciones
    auditoria = {
        'metodo': 'PCA ponderado sobre correlaciones; casos completos',
        'n_hogares': int(len(datos)),
        'n_casos_completos_peso_positivo': int(valido.sum()),
        'n_excluidos': int((~valido).sum()),
        'variables_utilizadas': vars_usables,
        'variables_sin_variacion': sorted(set(X.columns) - set(vars_usables)),
        'media_ponderada_original_por_variable': medias,
        'desviacion_ponderada_original_por_variable': desviaciones,
        'cargas_pc1': dict(zip(vars_usables, map(float, cargas))),
        'varianza_explicada_pc1': float(autovalores[0] / autovalores.sum()),
    }
    return scores, auditoria


def quintiles_ponderados(scores: pd.Series, pesos: pd.Series) -> pd.Series:
    s, w = numerica(scores), numerica(pesos)
    m = s.notna() & w.gt(0) & np.isfinite(w)
    out = pd.Series(np.nan, index=s.index, dtype='float64')
    if int(m.sum()) < 5:
        return out
    tabla = pd.DataFrame({'indice': s[m].to_numpy(dtype=float),
                          'peso': w[m].to_numpy(dtype=float)}, index=s.index[m])
    tabla = tabla.sort_values('indice', kind='mergesort')
    tabla['cuantil'] = (tabla['peso'].cumsum() - tabla['peso'] / 2) / tabla['peso'].sum()
    tabla['cuantil_empates'] = tabla.groupby('indice')['cuantil'].transform('mean')
    out.loc[tabla.index] = np.minimum(5, np.floor(tabla['cuantil_empates'] * 5) + 1)
    return out


## 7. Funciones complementarias de conectividad


In [ ]:
def agregar_cap500_a_hogares(personas: pd.DataFrame, cap500: pd.DataFrame,
                            auditoria: dict):
    d = incorporar_persona_objetivo(cap500, 'P500_A', 'CAP500', auditoria)
    verificar_unicidad(d, CLAVE_PERSONA, 'CAP500')
    d = d[CLAVE_PERSONA + ['P501', 'P502']]
    mayores6 = personas.loc[numerica(personas['P208_A']).ge(6), CLAVE_PERSONA].copy()
    a = mayores6.merge(d, on=CLAVE_PERSONA, how='left', validate='one_to_one',
                       indicator='_registro_cap500')
    a['_presente500'] = a['_registro_cap500'].eq('both').astype('int8')
    ultimo_mes = numerica(a['P502'])
    alguna_vez = numerica(a['P501'])
    a['_internet_mes'] = np.where(ultimo_mes.eq(1), 1.0,
                                  np.where(ultimo_mes.eq(2) |
                                           (ultimo_mes.eq(0) & alguna_vez.eq(2)), 0.0, np.nan))
    grupo = a.groupby(CLAVE_HOGAR, sort=False, dropna=False)
    r = grupo.agg(
        n_personas_6mas_cap500=('P201', 'size'),
        n_respuestas_cap500=('_presente500', 'sum'),
        n_uso_internet_mes_observado=('_internet_mes', 'count'),
        n_usuarios_internet_mes=('_internet_mes', lambda s: s.eq(1).sum()),
    ).reset_index()
    r['cobertura_cap500_completa'] = (
        (r['n_personas_6mas_cap500'] == r['n_respuestas_cap500']) &
        (r['n_personas_6mas_cap500'] == r['n_uso_internet_mes_observado'])
    ).astype('int8')
    r['n_usuarios_internet_mes'] = r['n_usuarios_internet_mes'].where(
        r['cobertura_cap500_completa'].eq(1), np.nan
    )
    r['prop_usuarios_internet_6mas'] = (
        r['n_usuarios_internet_mes'] / r['n_personas_6mas_cap500']
    )
    auditoria['uniones']['CAP500'] = {
        'residentes_6mas': int(len(mayores6)),
        'respuestas_enlazadas': int(a['_presente500'].sum()),
        'hogares_completos_cap500': int(r['cobertura_cap500_completa'].sum()),
    }
    return r


## 8. Exportación de una sola base Excel y validaciones


In [ ]:
def _valor_excel(valor):
    if valor is None:
        return None
    if isinstance(valor, (list, dict, tuple, set)):
        return json.dumps(valor, ensure_ascii=False, default=str)
    if pd.isna(valor):
        return None
    if isinstance(valor, (np.integer,)):
        return int(valor)
    if isinstance(valor, (np.floating,)):
        valor = float(valor)
    if isinstance(valor, (np.bool_,)):
        return bool(valor)
    if isinstance(valor, str):
        return "'" + valor if valor.startswith('=') else valor
    if isinstance(valor, pd.Timestamp):
        return valor.to_pydatetime()
    return valor


def _agregar_tabla_excel(libro: Workbook, tabla: pd.DataFrame,
                         titulo: str, config: Configuracion) -> list[str]:
    if tabla.columns.duplicated().any():
        raise ValueError(f'Excel {titulo}: existen columnas duplicadas.')
    if tabla.shape[1] > 16_384:
        raise ValueError(f'Excel {titulo}: mas de 16 384 columnas.')
    max_filas = int(config.excel_filas_maximas_por_hoja)
    if not 1 <= max_filas <= 1_048_575:
        raise ValueError('excel_filas_maximas_por_hoja debe estar entre 1 y 1048575.')

    columnas = list(map(str, tabla.columns))
    segmentos = max(1, (len(tabla) + max_filas - 1) // max_filas)
    nombres = []
    formatos_pct = {'prop_usuarios_internet_6mas', 'prop_mujeres'}
    formatos_3dec = {'indice_activos_pca', 'carga_pc1'}
    formatos_peso = {'factor_hogar', 'factor_suma_validos', 'coincidencias_ponderadas'}

    for i in range(segmentos):
        nombre = titulo if i == 0 else f'{titulo}_{i+1}'
        if len(nombre) > 31:
            raise ValueError(f'Nombre de hoja Excel demasiado largo: {nombre}')
        nombres.append(nombre)
        hoja = libro.create_sheet(title=nombre)
        hoja.sheet_view.showGridLines = False
        hoja.freeze_panes = 'F2' if all(c in columnas for c in CLAVE_HOGAR) else 'A2'
        hoja.sheet_properties.pageSetUpPr.fitToPage = True

        for j, col in enumerate(columnas, start=1):
            nombre_excel = get_column_letter(j)
            if col in {'descripcion', 'nota', 'detalle', 'fuente'}:
                ancho = 65
            elif col in {'programa', 'programa_1', 'programa_2', 'variable', 'indicador'}:
                ancho = 34
            elif col in CLAVE_HOGAR or col == 'hogar_id':
                ancho = min(26, max(14, len(col) + 5))
            else:
                ancho = min(32, max(14, len(col) + 2))
            hoja.column_dimensions[nombre_excel].width = ancho

        encabezado = []
        for col in columnas:
            celda = WriteOnlyCell(hoja, value=col)
            celda.font = Font(name='Aptos', size=10, bold=True, color='FFFFFF')
            celda.fill = PatternFill(fill_type='solid', fgColor='183A5B')
            celda.alignment = Alignment(horizontal='center', vertical='center',
                                        wrap_text=True)
            encabezado.append(celda)
        hoja.append(encabezado)

        inicio = i * max_filas
        fin = min(len(tabla), (i + 1) * max_filas)
        bloque = tabla.iloc[inicio:fin]
        formatear = {
            ix: ('0.0%' if c in formatos_pct else
                 '0.000' if c in formatos_3dec else '#,##0.00')
            for ix, c in enumerate(columnas)
            if c in formatos_pct | formatos_3dec | formatos_peso
        }
        for fila in bloque.itertuples(index=False, name=None):
            valores = [_valor_excel(v) for v in fila]
            for j, formato in formatear.items():
                if valores[j] is not None and isinstance(valores[j], (int, float)):
                    celda = WriteOnlyCell(hoja, value=valores[j])
                    celda.number_format = formato
                    valores[j] = celda
            hoja.append(valores)
        hoja.auto_filter.ref = f'A1:{get_column_letter(len(columnas))}{len(bloque)+1}'
        print(f'Excel: pestaña {nombre}: {len(bloque):,} registros')
    return nombres

def exportar_solo_excel(base_hogares: pd.DataFrame, config: Configuracion) -> Path:
    salida = Path(config.directorio_salida) if config.directorio_salida else Path(config.directorio_entrada)
    salida.mkdir(parents=True, exist_ok=True)
    archivo = salida / 'ENAPRES2025.xlsx'
    libro = Workbook(write_only=True)
    _agregar_tabla_excel(libro, base_hogares, 'Base_hogares', config)
    libro.save(archivo)
    print(f'\nEXCEL CREADO: {archivo.resolve()}')
    print(f'Hogares: {len(base_hogares):,} | Variables: {base_hogares.shape[1]:,}')
    return archivo


In [ ]:
def chequear_base(base: pd.DataFrame, auditoria: dict) -> None:
    verificar_unicidad(base, CLAVE_HOGAR, 'BASE FINAL HOGARES')
    if not (base['tamano_hogar'] >= 1).all():
        raise ValueError('Hay hogares sin miembros residentes validos')
    if 'prog_beca_18' in base.columns or 'P306_5' in base.columns:
        raise ValueError('Se encontro accidentalmente Beca 18')
    for n in NODOS:
        errores = base[n].notna() & ~base[n].isin([0, 1])
        if errores.any():
            raise ValueError(f'El nodo {n} presenta valores diferentes de 0/1/NA')
    n_eligibles = int(base[NODOS].notna().all(axis=1).sum())
    auditoria['validaciones'] = {
        'hogares_finales': int(len(base)),
        'nodos_ising': NODOS,
        'hogares_con_todos_nodos_observados': n_eligibles,
        'hogares_sin_todos_nodos': int(len(base) - n_eligibles),
        'hogares_sin_factor_positivo': int((~numerica(base['factor_hogar']).gt(0)).sum()),
        'hogares_sin_indice_pca': int(base['indice_activos_pca'].isna().sum()),
        'hogares_urbanos': int(base['area'].eq(1).sum()),
        'hogares_rurales': int(base['area'].eq(2).sum()),
        'jefes_sin_lengua_materna': int(base['lengua_materna_jefe'].isna().sum()),
    }


In [ ]:
def ejecutar_y_generar_excel(config: Configuracion):
    auditoria = {
        'lectura': [], 'filtros': {}, 'personas_objetivo': {},
        'uniones': {}, 'activos': {}, 'pca': {}, 'validaciones': {},
    }
    print('1/5 Leyendo padrón CAP200 y definiendo residentes habituales...')
    cap200 = cargar_capitulo(config, '200', auditoria)
    padron, hogares = construir_padron(cap200, config, auditoria)
    del cap200

    print('2/5 Agregando programas sociales, discapacidad, educación e identidad...')
    cap300 = cargar_capitulo(config, '300C', auditoria)
    hogares_prog, personas_combinadas = agregar_cap300_a_hogares(
        padron, cap300, hogares, config, auditoria
    )
    del cap300, personas_combinadas
    hogares = hogares.merge(hogares_prog, on=CLAVE_HOGAR, how='left',
                            validate='one_to_one', sort=False)
    if len(hogares) != len(hogares_prog):
        raise ValueError('La agregación CAP300C no coincide con hogares CAP200.')
    del hogares_prog

    print('3/5 Agregando vivienda, agua, saneamiento, electricidad, conectividad y PCA de bienes...')
    cap100e = cargar_capitulo(config, '100E', auditoria)
    hogares_vivienda = construir_hogares_cap100e(cap100e, auditoria)
    del cap100e
    hogares = hogares.merge(hogares_vivienda, on=CLAVE_HOGAR, how='left',
                            validate='one_to_one', sort=False)
    auditoria['uniones']['CAP100E'] = {
        'hogares_master_cap200': int(len(hogares)),
        'hogares_sin_cap100e': int(hogares['factor_hogar'].isna().sum()),
    }
    del hogares_vivienda

    print('4/5 Consolidando uso de internet de personas de 6 años o más...')
    if config.incluir_modulo_500:
        cap500 = cargar_capitulo(config, '500', auditoria)
        hogares500 = agregar_cap500_a_hogares(padron, cap500, auditoria)
        del cap500
        hogares = hogares.merge(hogares500, on=CLAVE_HOGAR, how='left',
                                validate='one_to_one', sort=False)
        del hogares500
    del padron

    hogares['hogar_id'] = hogares[CLAVE_HOGAR].astype('string').agg('_'.join, axis=1)
    hogares['densidad_personas_por_habitacion'] = (
        hogares['tamano_hogar'] / hogares['habitaciones_vivienda'].where(
            hogares['habitaciones_vivienda'].gt(0))
    )
    coincidencia_area = hogares['area'].notna() & hogares['area_cap100e'].notna()
    auditoria['uniones']['desacuerdo_area_cap200_cap100e'] = int((
        coincidencia_area & hogares['area'].ne(hogares['area_cap100e'])
    ).sum())
    chequear_base(hogares, auditoria)

    ausentes = sorted(set(COLUMNAS_FINALES) - set(hogares.columns))
    if ausentes:
        raise ValueError(f'No se pudieron construir estas variables: {ausentes}')
    hogares = hogares.loc[:, COLUMNAS_FINALES].copy()
    if hogares.shape[1] != 35:
        raise AssertionError('Se esperaban exactamente 35 columnas')
    print('5/5 Exportando ÚNICAMENTE el archivo Excel compacto (35 columnas)...')
    archivo_excel = exportar_solo_excel(hogares, config)
    return hogares, archivo_excel, auditoria


## 9. Configuración de rutas y ejecución


In [ ]:
RUTA_SAV = Path('/mnt/data') if Path('/mnt/data').exists() else Path.cwd()

configuracion = Configuracion(
    directorio_entrada=RUTA_SAV,
    directorio_salida=RUTA_SAV,
    resfin_permitidos=(1,),
    solo_urbano=False,
    incluir_modulo_500=False,
)

base_hogares, ruta_excel, auditoria = ejecutar_y_generar_excel(configuracion)
print(f"\nARCHIVO EXCEL: {ruta_excel}")


## 10. Verificación opcional


In [ ]:
print('Archivo Excel:', ruta_excel)
print('Total de hogares:', f'{len(base_hogares):,}')
print('Total de variables:', base_hogares.shape[1], '(7 programas + 22 controles + 6 técnicas)')
print('Hogares con 7 nodos observados:', f"{auditoria['validaciones']['hogares_con_todos_nodos_observados']:,}")
display(base_hogares.head(5))
